# Net Sales Gold Table — `fact_sales`

Wires `dim_gross_price` into the month-grain `fact_orders` and builds the standard FMCG value waterfall:

`gross_sales` → − pre-invoice deductions → `net_invoice_sales` → − post-invoice deductions → `net_sales`

Grain stays the same as `fact_orders`: **month × product_code × customer_code**.

> Run the cells top to bottom. First run creates the table; reruns MERGE on the natural key, so it's safe to re-run.

### Imports

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

### Parameters

`catalog` / `gold_schema` match your existing setup. The two deduction percentages are **placeholder business rules** (set via widgets) — see the production note at the bottom for how to replace them with real deduction tables.

In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("gold_schema", "gold", "Gold Schema")
dbutils.widgets.text("pre_invoice_pct", "0.18", "Pre-invoice deduction %")
dbutils.widgets.text("post_invoice_pct", "0.08", "Post-invoice deduction %")

catalog          = dbutils.widgets.get("catalog")
gold_schema      = dbutils.widgets.get("gold_schema")
pre_invoice_pct  = float(dbutils.widgets.get("pre_invoice_pct"))
post_invoice_pct = float(dbutils.widgets.get("post_invoice_pct"))

fact_orders_table = f"{catalog}.{gold_schema}.fact_orders"
price_table       = f"{catalog}.{gold_schema}.dim_gross_price"
target_table      = f"{catalog}.{gold_schema}.fact_sales"

print("Source fact :", fact_orders_table)
print("Price dim   :", price_table)
print("Target      :", target_table)
print("Pre / Post %:", pre_invoice_pct, post_invoice_pct)

### Step 1 — Load the fact and the price dimension

`fact_orders.date` is the first day of each month. `dim_gross_price` is keyed by `product_code` + `year` (`year` is stored as a **string**).

In [0]:
df_fact  = spark.table(fact_orders_table)
df_price = spark.table(price_table)

print("fact_orders rows:", df_fact.count())
df_fact.show(5)
df_price.show(5)

### Step 2 — Attach `gross_price`

Derive the order year from the fact date and join price on `product_code` + `year`. A **LEFT** join keeps every order even when a price is missing; unmatched rows fall back to `0` and are flagged so revenue is never silently dropped.

In [0]:
# dim_gross_price.year is a string -> compare year-on-year as strings
df_fact_y = df_fact.withColumn("order_year", F.year("date").cast("string"))

df_priced = (
    df_fact_y.alias("f")
    .join(
        df_price.alias("p"),
        on=[
            F.col("f.product_code") == F.col("p.product_code"),
            F.col("f.order_year") == F.col("p.year").cast("string"),
        ],
        how="left",
    )
    .withColumn("price_missing", F.col("p.price_inr").isNull())
    .withColumn("gross_price", F.coalesce(F.col("p.price_inr").cast("double"), F.lit(0.0)))
    .select(
        F.col("f.date").alias("date"),
        F.col("f.product_code").alias("product_code"),
        F.col("f.customer_code").alias("customer_code"),
        F.col("f.sold_quantity").alias("sold_quantity"),
        F.col("gross_price"),
        F.col("price_missing"),
    )
)

missing = df_priced.filter("price_missing").count()
print("Rows without a matching price:", missing)
df_priced.show(5)

### Step 3 — Build the net-sales waterfall

`gross_sales = sold_quantity × gross_price`, then subtract pre- and post-invoice deductions in sequence.

In [0]:
df_sales = (
    df_priced
    .withColumn("gross_sales", F.round(F.col("sold_quantity") * F.col("gross_price"), 2))
    .withColumn("pre_invoice_deductions", F.round(F.col("gross_sales") * F.lit(pre_invoice_pct), 2))
    .withColumn("net_invoice_sales", F.round(F.col("gross_sales") - F.col("pre_invoice_deductions"), 2))
    .withColumn("post_invoice_deductions", F.round(F.col("net_invoice_sales") * F.lit(post_invoice_pct), 2))
    .withColumn("net_sales", F.round(F.col("net_invoice_sales") - F.col("post_invoice_deductions"), 2))
    .select(
        "date", "product_code", "customer_code", "sold_quantity",
        "gross_price", "gross_sales",
        "pre_invoice_deductions", "net_invoice_sales",
        "post_invoice_deductions", "net_sales",
    )
)

df_sales.show(5)

### Step 4 — Write to gold (create or MERGE)

Same idempotent pattern as your fact load notebooks: create on first run, MERGE on the natural key thereafter.

In [0]:
if not spark.catalog.tableExists(target_table):
    print("Creating new table:", target_table)
    (
        df_sales.write
        .format("delta")
        .option("delta.enableChangeDataFeed", "true")
        .mode("overwrite")
        .saveAsTable(target_table)
    )
else:
    print("Merging into existing table:", target_table)
    tgt = DeltaTable.forName(spark, target_table)
    (
        tgt.alias("t")
        .merge(
            df_sales.alias("s"),
            "t.date = s.date AND t.product_code = s.product_code AND t.customer_code = s.customer_code",
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

### Step 5 — Sanity check + sample KPI

Monthly gross vs net sales — the kind of trend your AI/BI dashboard or Genie space would chart.

In [0]:
df_gold = spark.table(target_table)
print("fact_sales rows:", df_gold.count())

(
    df_gold.groupBy("date")
    .agg(
        F.round(F.sum("gross_sales"), 2).alias("gross_sales"),
        F.round(F.sum("net_sales"), 2).alias("net_sales"),
    )
    .orderBy("date")
    .show(24)
)

---
### Production note — making the deductions real

The two flat percentages above are placeholders so the pipeline runs end-to-end today. In a real FMCG model, pre- and post-invoice deductions live in their own **source tables** and join in as dimensions:

- `pre_invoice_deductions` → `pre_invoice_discount_pct` per **customer × product × year**
- `post_invoice_deductions` → `discounts_pct` + `other_deductions_pct` per **customer × product × month**

Swap the two rate `withColumn` lines in Step 3 for left joins to those tables, e.g.:

```python
# df_priced = df_priced.join(df_pre_inv, ["customer_code", "product_code", "order_year"], "left")
# .withColumn("pre_invoice_deductions",
#             F.round(F.col("gross_sales") * F.coalesce("pre_invoice_discount_pct", F.lit(0.0)), 2))
```

**Also worth fixing upstream:** in `3_pricing_data_processing`, the MERGE into `dim_gross_price` matches only on `product_code`, so it collapses to one row per product and loses the per-year price grain. Add `AND target.year = source.year` to the merge condition so 2024 and 2025 prices both survive — this join in Step 2 depends on that grain.